# Two-qubit Quantum Algorithms

In the past three modules you have acquired the expertise to run NMR experiments, determine the Hamiltonian, perform single-qubit and two-qubit gates, and crucially how to initiate the quantum system in a pseudo-pure state. All of these will be used in this module to run quantum algorithms. It is crucial that you have a clear idea what pulses need to be applied, so we strongly recommend that you simulate the experiment using QuTiP, before running it on the spectrometer.

In [1]:
# Initialize Python libraries
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

## Two-qubit Deutsch-Jozsa algorithm

Deutsch-Jozsa algorithm was the first quantum algorithm found to perform better than its best counterpart. It determines if a Boolean function is constant or balanced. 

In order to perform this algorith in a reversible fashion we use two-qubit state $|x \hspace{0.25em}y\rangle$ initialized in $|\psi_0\rangle=|00\rangle$. The first step is to perform $U_1= (\pi/2I_y^2)(-\pi/2I_y^1)$ simultaneous rotations on both qubits. The resulting state is

$$
\begin{aligned} 
|\psi_1\rangle=U_1|\psi_0\rangle= \frac{|0\rangle+|1\rangle}{\sqrt{2}}\otimes\frac{|0\rangle-|1\rangle}{\sqrt{2}}\\
=|00\rangle -|01\rangle +|10\rangle-|11\rangle 
\end{aligned}
$$

At this point, the unitary transform $U_f$ that implements actions of function $f$ is applied. To find corresponding unitaries, note that we use $U_f|x\hspace{0.25em}y\rangle=|x\hspace{0.5em}f(x)\oplus y\rangle$. The reason can be seen by writing the qubit state after the unitary transform $|\psi_2\rangle=U_f|\psi_1\rangle$.
$$
\begin{aligned}
|\psi_2\rangle&=\frac{1}{2}\{|0 \hspace{0.5em}0\rangle- |0 \hspace{0.5em}f(0)\oplus1\rangle+|1 \hspace{0.5em}f(1)\rangle-|1 \hspace{0.5em}f(1)\oplus1\rangle\}\\
&=\frac{1}{2}\{ (-1)^{f(0)} |0\rangle(|0\rangle-|1\rangle)+(-1)^{f(1)} |1\rangle(|0\rangle-|1\rangle)\}\\
&=\frac{(-1)^{f(0)}|0\rangle+(-1)^{f(1)}|1\rangle}{\sqrt{2}}\otimes\frac{|0\rangle-|1\rangle}{\sqrt{2}}
\end{aligned}
$$

Note that the state of qubit 1, on the right, remains unchanged. The remaining step is to invert the initial rotations using $U_3=(-\pi/2I_y^2)(\pi/2I_y^1)$.
$$
\begin{aligned}
|\psi_3\rangle&=U_3|\psi_2\rangle\\
&= \frac{1}{2}\{(-1)^{f(0)}(|0\rangle-|1\rangle)+(-1)^{f(1)}(|0\rangle+|1\rangle)\}\otimes |0\rangle\\
&= \frac{1}{2} \{((-1)^{f(0)}+(-1)^{f(1)})|0\rangle+(-(-1)^{f(0)}+(-1)^{f(1)})|1\rangle\}\otimes |0\rangle
\end{aligned}
$$

Again, we see that the qubit number 1 is left unchanged and the state of qubit 2 reflects wether f is constant or balanced.

So, with the right implementation of the function as a unitary, the pulse sequence for the Deutsch-Jozsa algorithm is $U_3.U_f.U_1|00\rangle$.

To find the unitaries recall that for two classical bits, there exist four possible boolean functions

|x|f1(x)|f2(x)|f3(x)|f4(x)|
| :---- | :-: | :-: | :-: | :-:
| 0 | 0 | 1 | 0 | 1 |
| 1 | 0 | 1 | 1 | 0 |

Where the first two cases are constant, and the last two are balanced functions.

You have worked on the unitaries in your prelab exercise, and the result should be operations that you already know how to implement. You should have already simulated the whole experiment, i.e. 3 initial states, to achieve temporal averaging for the pseudo-pure state and four experiments to implement $U_f$ unitaries. 

Perform the NMR experiments for 12 cases needed for Deutsch-Jozsa algorith.

Notes:
- Be mindful of required repetition delays due to $T_1$
- When applying simultaneous pulses, the duration of the pulse for both 1H and 13C channels, needs to be equal.
- Calibrate pulse amplitude levels for the CNOT gate, to the best of your ability
- The results for the first two unitaries should be the same, and different from the last two unitaries.

## Quantum Simulation of Ground-State Energy of H$_2$

Below is a set of instructions corresponding to the three main experimental steps, adapted from the procedures described in \textit{NMR Implementation of a Molecular Hydrogen Quantum Simulation with Adiabatic State Preparation} (PRL 104, 030502, 2010).

Step 1: Preparation of the Pseudopure State:
As in the previous experiment, start by preparing the pseudopure state corresponding to the $\vert 00 \rangle$ state.

Step 2: Adiabatic State Preparation (ASP):
Start by preparing the system qubit in the ground state of a simple Hamiltonian, for example, $    H_0 = -\sigma_x,$ whose ground state is $    |\psi_0\rangle = \frac{1}{\sqrt{2}} (|0\rangle - |1\rangle).$

Hamiltonian Mapping: The target Hamiltonian $H$ represents the molecular Hamiltonian for the hydrogen molecule in the minimal STO-3G basis. Map the two relevant electronic configurations onto a 2$\times$2 Hamiltonian.

Linear Interpolation: Evolve the system from $H_0$ to $H$ by slowly varying the Hamiltonian using the interpolation $  H(s) = (1-s)H_0 + sH, \quad s = \frac{t}{T},$
where $t$ runs from 0 to a total evolution time $T$. In practice, discretize this evolution into several steps (with $s_m = \frac{m}{M+1}$ for each step $m$) and implement each small evolution step by the unitary operator $    U_{ad}^{(m)} = \exp\Bigl(-i\,H(s_m)\Delta t\Bigr), \quad \Delta t = \frac{T}{M+1}.$

 Ensure that $T$ is chosen long enough to satisfy the adiabatic condition so that the system qubit remains in the instantaneous ground state, finally reaching the ground state of $H$.


Step 3: Controlled Unitary Operation \& Iterative Phase Estimation:

Probe Qubit Preparation: Prepare the probe qubit (e.g., the $^{1}$H nucleus) in the superposition state $    |+\rangle = \frac{1}{\sqrt{2}} (|0\rangle + |1\rangle)$  using a pseudo-Hadamard gate.

Controlled Operation: Implement the controlled unitary operation for the first iteration using $
    U_0 = e^{-iH\tau},$    so that the controlled-$U_0$ gate is $    C\text{-}U_0 = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes U_0.$     When applied, this gate imprints a phase $e^{-iE\tau}$ on the probe qubit (with $E$ being the ground-state energy of $H$).

Iterative Scheme: To achieve high-precision phase estimation, use an iterative method. In each iteration $k$, modify the controlled operation by feeding back the previously measured phase. For example, the next iteration applies $    U_{k+1} = \left(e^{-i2\pi\phi_0^{(k)}} U_k\right)^{2^n},$ 
    where $n$ (typically 3) is the number of bits extracted per iteration, and $\phi_0^{(k)}$ is the correction derived from the $k^\text{th}$ measurement.
    
Measurement: Measure the phase shift using NMR interferometry by analyzing the Fourier-transformed spectrum of the probe qubit. Each measured phase is used to refine the phase estimate until the desired precision is reached.


Please refer to the original paper for detailed pulse sequences, parameter values, and further experimental nuances. Follow the steps as explained in the publication to ensure accurate replication of the experiment.


